<a href="https://colab.research.google.com/github/gns1719/Pet-NosePrint-Id-Service/blob/Jun/dohyeon_butack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

In [ ]:
from google.colab import files
import zipfile
import os

# 1. zip 파일 업로드
uploaded = files.upload()  # 파일 업로드 창이 열립니다

# 2. 업로드한 파일 이름 가져오기
filename = list(uploaded.keys())[0]  # 첫 번째 업로드 파일 이름

# 3. 압축 해제할 디렉토리 설정 (원하는 경우 변경 가능)
extract_dir = 'nose_dog'

# 디렉토리 없으면 생성
os.makedirs(extract_dir, exist_ok=True)

# 4. zip 파일 압축 해제
with zipfile.ZipFile(filename, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"압축 해제 완료: {extract_dir} 폴더에 저장됨")

Saving nose_filtered1.zip to nose_filtered1.zip
압축 해제 완료: nose_dog 폴더에 저장됨


In [ ]:
import os
import cv2
import csv
import random
from glob import glob
from skimage.feature import local_binary_pattern

# 원본 이미지 경로
source_dir = '/content/nose_dog'

# 저장 폴더
temp_aug_dir = '/content/nose_test/triplet_images'
os.makedirs(temp_aug_dir, exist_ok=True)

# 각 CSV 파일 경로
triplet_csv_path = '/content/nose_test/triplets.csv'
classification_csv_path = '/content/nose_test/classification.csv'
pair_csv_path = '/content/nose_test/pairs.csv'

rotation_angles = [0, 45, 90, 135, 180, 225, 270, 315]

def compute_lbp(image, P=8, R=1):
    lbp = local_binary_pattern(image, P, R, method='uniform')
    return lbp.astype('uint8')

# 모든 이미지 로드
image_paths = sorted(glob(os.path.join(source_dir, '*.jpg')))

# 클래스별 이미지 회전 후 저장, 클래스 라벨 부여
class_images = {}  # {class_name: [list of augmented image paths]}
class_labels = {}  # {image_path: class_label (int)}

for idx, path in enumerate(image_paths):
    class_name = f'dog{idx+1}'
    class_images[class_name] = []

    img = cv2.imread(path)
    if img is None:
        continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    for angle in rotation_angles:
        h, w = gray.shape
        center = (w//2, h//2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_LANCZOS4, borderMode=cv2.BORDER_REFLECT)
        lbp = compute_lbp(rotated)

        save_name = f'{class_name}_{angle}.png'
        save_path = os.path.join(temp_aug_dir, save_name)
        cv2.imwrite(save_path, lbp)

        class_images[class_name].append(save_path)
        class_labels[save_path] = idx  # 정수형 라벨 (0부터 시작)

# --- CSV 작성 시작 ---

# 1. Triplet Loss용 CSV
with open(triplet_csv_path, 'w', newline='') as f_triplet:
    writer_triplet = csv.writer(f_triplet)
    writer_triplet.writerow(['anchor', 'positive', 'negative'])

    for class_name, imgs in class_images.items():
        if len(imgs) < 2:
            continue

        for i in range(len(imgs)):
            anchor_img = imgs[i]
            positive_candidates = [img for j, img in enumerate(imgs) if j != i]
            positive_img = random.choice(positive_candidates)

            # negative는 다른 클래스에서 무작위 선택
            negative_class = random.choice([c for c in class_images if c != class_name])
            negative_img = random.choice(class_images[negative_class])

            writer_triplet.writerow([anchor_img, positive_img, negative_img])

# 2. Classification용 CSV (image_path, label)
with open(classification_csv_path, 'w', newline='') as f_cls:
    writer_cls = csv.writer(f_cls)
    writer_cls.writerow(['image_path', 'label'])
    for img_path, label in class_labels.items():
        writer_cls.writerow([img_path, label])

# 3. Pair-wise Circle Loss용 CSV (img1, img2, pair_label)
with open(pair_csv_path, 'w', newline='') as f_pair:
    writer_pair = csv.writer(f_pair)
    writer_pair.writerow(['img1', 'img2', 'pair_label'])

    all_images = []
    for imgs in class_images.values():
        all_images.extend(imgs)

    # positive pairs: 같은 클래스 내에서 랜덤 쌍 생성
    for class_name, imgs in class_images.items():
        if len(imgs) < 2:
            continue
        for i in range(len(imgs)):
            for j in range(i+1, len(imgs)):
                writer_pair.writerow([imgs[i], imgs[j], 1])  # positive pair

    # negative pairs: 다른 클래스 이미지에서 랜덤 샘플링 쌍 생성 (갯수는 positive 쌍과 비슷하게 맞추면 됨)
    num_positive_pairs = sum(len(imgs)*(len(imgs)-1)//2 for imgs in class_images.values())
    neg_pairs_created = 0
    while neg_pairs_created < num_positive_pairs:
        img1 = random.choice(all_images)
        img2 = random.choice(all_images)
        if class_labels[img1] != class_labels[img2]:
            writer_pair.writerow([img1, img2, 0])  # negative pair
            neg_pairs_created += 1

print("✅ Triplet, Classification, Pair-wise CSV가 모두 생성되었습니다.")

✅ Triplet, Classification, Pair-wise CSV가 모두 생성되었습니다.


In [ ]:
pip install tqdm

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import csv
import numpy as np
from tqdm import tqdm
import random

# ==================== 재현성 고정 ==================== #
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

# ==================== 하이퍼파라미터 ==================== #
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-4
IMG_SIZE = 224
ALPHA = 0.5
BETA = 0.5

# CSV 경로
TRIPLET_CSV = '/content/nose_test/triplets.csv'
CLASS_CSV = '/content/nose_test/classification.csv'
PAIR_CSV = '/content/nose_test/pairs.csv'
SAVE_DIR = '/content/triplet_multi_loss_models'
os.makedirs(SAVE_DIR, exist_ok=True)

# ==================== Dataset 정의 ==================== #
class TripletDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.triplets = []
        with open(csv_path, 'r') as f:
            reader = csv.reader(f)
            next(reader)
            for row in reader:
                self.triplets.append(row)
        self.transform = transform

    def __getitem__(self, idx):
        anchor_path, positive_path, negative_path = self.triplets[idx]
        anchor_class = self.get_class_id(anchor_path)
        positive_class = self.get_class_id(positive_path)
        negative_class = self.get_class_id(negative_path)

        anchor = self.preprocess_lbp(anchor_path)
        positive = self.preprocess_lbp(positive_path)
        negative = self.preprocess_lbp(negative_path)

        return anchor, positive, negative, anchor_class, positive_class, negative_class

    def __len__(self):
        return len(self.triplets)

    def get_class_id(self, path):
        name = os.path.basename(path)
        return int(name.split('_')[0].replace('dog', '')) - 1

    def preprocess_lbp(self, image_path):
        img = Image.open(image_path).convert('L')
        img_np = np.array(img)
        lbp = self.compute_lbp(img_np)
        lbp_pil = Image.fromarray(lbp).convert('RGB')
        return self.transform(lbp_pil) if self.transform else lbp_pil

    def compute_lbp(self, image, P=8, R=1):
        from skimage.feature import local_binary_pattern
        lbp = local_binary_pattern(image, P, R, method='uniform')
        return ((lbp / lbp.max()) * 255).astype(np.uint8)

class PairDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.pairs = []
        with open(csv_path, 'r') as f:
            reader = csv.reader(f)
            next(reader)
            for row in reader:
                self.pairs.append(row)
        self.transform = transform

    def __getitem__(self, idx):
        img1_path, img2_path, label = self.pairs[idx]
        label = int(label)
        img1 = self.preprocess_lbp(img1_path)
        img2 = self.preprocess_lbp(img2_path)
        return img1, img2, label

    def __len__(self):
        return len(self.pairs)

    def preprocess_lbp(self, image_path):
        img = Image.open(image_path).convert('L')
        img_np = np.array(img)
        lbp = self.compute_lbp(img_np)
        lbp_pil = Image.fromarray(lbp).convert('RGB')
        return self.transform(lbp_pil) if self.transform else lbp_pil

    def compute_lbp(self, image, P=8, R=1):
        from skimage.feature import local_binary_pattern
        lbp = local_binary_pattern(image, P, R, method='uniform')
        return ((lbp / lbp.max()) * 255).astype(np.uint8)

# ==================== 전처리 ==================== #
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

triplet_dataset = TripletDataset(TRIPLET_CSV, transform=transform)
pair_dataset = PairDataset(PAIR_CSV, transform=transform)

triplet_loader = DataLoader(triplet_dataset, batch_size=BATCH_SIZE, shuffle=True)
pair_loader = DataLoader(pair_dataset, batch_size=BATCH_SIZE, shuffle=True)

# ==================== 모델 정의 ==================== #
class TripletMultiLossNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = models.resnet18(pretrained=True)
        self.backbone.fc = nn.Identity()
        self.embedding = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        embed = self.embedding(features)
        logits = self.classifier(embed)
        return embed, logits

# ==================== Loss 정의 ==================== #
class CircleLoss(nn.Module):
    def __init__(self, margin=0.25, gamma=256):
        super().__init__()
        self.margin = margin
        self.gamma = gamma

    def forward(self, embed1, embed2, label):
        sim = F.cosine_similarity(embed1, embed2)
        pos_mask = (label == 1).float()
        neg_mask = (label == 0).float()
        ap = torch.clamp_min(-sim + 1 + self.margin, 0.) * pos_mask
        an = torch.clamp_min(sim + self.margin, 0.) * neg_mask
        loss = torch.log1p(torch.exp(self.gamma * (ap * sim + an * sim)))
        return loss.mean()

class TripletLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        d_pos = F.pairwise_distance(anchor, positive)
        d_neg = F.pairwise_distance(anchor, negative)
        return F.relu(d_pos - d_neg + self.margin).mean()

# ==================== 학습 준비 ==================== #
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes = len(set([triplet_dataset.get_class_id(x[0]) for x in triplet_dataset.triplets]))

model = TripletMultiLossNet(num_classes=num_classes).to(device)
criterion_triplet = TripletLoss()
criterion_ce = nn.CrossEntropyLoss()
criterion_circle = CircleLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ==================== 학습 루프 ==================== #
for epoch in range(EPOCHS):
    model.train()
    total_loss = total_triplet = total_ce = total_circle = 0
    correct_cls = total_cls = 0

    triplet_iter = iter(triplet_loader)
    pair_iter = iter(pair_loader)
    min_len = min(len(triplet_loader), len(pair_loader))
    pbar = tqdm(range(min_len), desc=f"Epoch {epoch+1}/{EPOCHS}")

    for _ in pbar:
        anchor, positive, negative, a_cls, p_cls, n_cls = next(triplet_iter)
        img1, img2, label = next(pair_iter)

        anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
        a_cls, p_cls, n_cls = a_cls.to(device), p_cls.to(device), n_cls.to(device)
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)

        optimizer.zero_grad()

        a_embed, a_logits = model(anchor)
        p_embed, p_logits = model(positive)
        n_embed, n_logits = model(negative)

        loss_triplet = criterion_triplet(a_embed, p_embed, n_embed)
        loss_ce = (
            criterion_ce(a_logits, a_cls) +
            criterion_ce(p_logits, p_cls) +
            criterion_ce(n_logits, n_cls)
        )

        with torch.no_grad():
            preds = torch.cat([a_logits, p_logits, n_logits]).argmax(dim=1)
            labels = torch.cat([a_cls, p_cls, n_cls])
            correct_cls += (preds == labels).sum().item()
            total_cls += preds.size(0)

        e1, _ = model(img1)
        e2, _ = model(img2)
        loss_circle = criterion_circle(e1, e2, label)

        loss = loss_triplet + ALPHA * loss_ce + BETA * loss_circle
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_triplet += loss_triplet.item()
        total_ce += loss_ce.item()
        total_circle += loss_circle.item()

        pbar.set_postfix({
            "Total": f"{loss.item():.4f}",
            "Triplet": f"{loss_triplet.item():.4f}",
            "CE": f"{loss_ce.item():.4f}",
            "Circle": f"{loss_circle.item():.4f}"
        })

    acc_cls = correct_cls / total_cls * 100
    print(f"\n✅ Epoch [{epoch+1}/{EPOCHS}] Finished.")
    print(f"📉 Avg Loss: {total_loss/min_len:.4f}, Triplet: {total_triplet/min_len:.4f}, CE: {total_ce/min_len:.4f}, Circle: {total_circle/min_len:.4f}")
    print(f"🎯 Classification Accuracy: {acc_cls:.2f}%")

    # ==================== 모델 저장 ==================== #
    save_path = os.path.join(SAVE_DIR, f'model_epoch{epoch+1}.pth')
    torch.save(model.state_dict(), save_path)
    print(f"💾 모델 저장됨: {save_path}\n")


Epoch 1/10: 100%|██████████| 2024/2024 [38:09<00:00,  1.13s/it, Total Loss=nan, Triplet=nan, CE=nan, Circle=nan]



Epoch [1/10] Finished. Avg Loss: nan | Triplet: nan, CE: nan, Circle: nan



Epoch 2/10: 100%|██████████| 2024/2024 [38:21<00:00,  1.14s/it, Total Loss=nan, Triplet=nan, CE=nan, Circle=nan]



Epoch [2/10] Finished. Avg Loss: nan | Triplet: nan, CE: nan, Circle: nan



Epoch 3/10: 100%|██████████| 2024/2024 [38:23<00:00,  1.14s/it, Total Loss=nan, Triplet=nan, CE=nan, Circle=nan]


...

Epoch 6/10: 100%|██████████| 2024/2024 [39:02<00:00,  1.16s/it, Total Loss=nan, Triplet=nan, CE=nan, Circle=nan]



Epoch [6/10] Finished. Avg Loss: nan | Triplet: nan, CE: nan, Circle: nan



Epoch 7/10: 100%|██████████| 2024/2024 [38:53<00:00,  1.15s/it, Total Loss=nan, Triplet=nan, CE=nan, Circle=nan]



Epoch [7/10] Finished. Avg Loss: nan | Triplet: nan, CE: nan, Circle: nan



Epoch 8/10:   0%|          | 10/2024 [00:11<37:07,  1.11s/it, Total Loss=nan, Triplet=nan, CE=nan, Circle=nan]